# Self-Matching Evaluation: Othello World

This notebook documents the consistency evaluation for the Othello World research project at `/net/scratch2/smallyan/othello_world_eval`.

## Project Overview
The project investigates whether language models trained on sequence prediction tasks develop internal representations of the underlying process generating sequences, using a GPT variant trained on predicting legal moves in Othello as a testbed.

## CS1: Conclusion vs Original Results

### Analysis

The plan.md contains claims based on the original Li et al. paper about probe accuracy. The main implementation notebook (Othello_GPT_Circuits.ipynb) is by Neel Nanda who extended the original work.

**Key Discrepancy Found:**

| Source | Claim about Linear Probes |
|--------|--------------------------|
| plan.md | "Linear probes never dip below 20% error across all layers" |
| Othello_GPT_Circuits.ipynb (Cell 66) | Linear probe achieves ~0.4% average error rate |

**Detailed Comparison:**

1. **Plan.md Hypothesis 3**: "Nonlinear probes are necessary to decode board state from internal activations, while linear probes fail."

2. **Notebook Finding (Cell 0)**: "I found that the network actually learns a **linear** world model"

3. **Numerical Results from Cell 66 Plotly Output**:
   - Black to Play: 0.37% average error, 2.48% max error
   - All Moves: 0.36% average error, 2.49% max error

The discrepancy arises because the notebook uses a different framing (my color vs their color) rather than absolute (black vs white), which makes linear probes work effectively.

In [ ]:
# Verification: Extract error rates from notebook outputs
import numpy as np

# Error rates extracted from Cell 66 Plotly heatmap data
black_to_play_errors = [
    [0.012000024,0.008000016,0.0055999756,0.0,0.0,0.0055999756,0.024800003,0.014400005],
    [0.0031999946,0.00080001354,0.0031999946,0.0,0.0,0.00080001354,0.00080001354,0.00880003],
    [0.00080001354,0.0048000216,0.0031999946,0.006399989,0.0016000271,0.0,0.0072000027,0.0055999756],
    [0.00080001354,0.0,0.004000008,0.00080001354,0.0031999946,0.004000008,0.0,0.0],
    [0.0,0.0,0.00880003,0.0072000027,0.002399981,0.0072000027,0.0,0.0],
    [0.0,0.002399981,0.0031999946,0.0031999946,0.0,0.002399981,0.00080001354,0.0],
    [0.011200011,0.0016000271,0.0,0.0,0.0,0.002399981,0.006399989,0.0],
    [0.00880003,0.010399997,0.0055999756,0.0016000271,0.0,0.002399981,0.002399981,0.014400005]
]

black_errors = np.array(black_to_play_errors)
print(f"Linear Probe Error Rates (Black to Play):")
print(f"  Average: {black_errors.mean()*100:.2f}%")
print(f"  Max: {black_errors.max()*100:.2f}%")
print(f"  Min: {black_errors.min()*100:.2f}%")
print()
print(f"Plan.md Claim: Linear probes never dip below 20% error")
print(f"Actual Result: {black_errors.mean()*100:.2f}% average error")
print()
print("VERDICT: MISMATCH - Plan claims >20% error, notebook shows <1% error")

### CS1 Verdict: **FAIL**

The plan's claim that "linear probes never dip below 20% error" directly contradicts the notebook's recorded results showing ~0.4% error rate for linear probes.

## CS2: Implementation Follows the Plan

### Plan Methodology Steps

1. Train an 8-layer GPT model (Othello-GPT) with 8-head attention and 512-dimensional hidden space
2. Use two datasets: championship and synthetic
3. Train nonlinear probes (2-layer MLPs) on internal activations
4. Perform interventional experiments using gradient descent
5. Create latent saliency maps

In [ ]:
# Verification: Check implementation of each step
import os

repo_path = '/net/scratch2/smallyan/othello_world_eval'

implementation_status = {}

# Step 1: Check model architecture
with open(os.path.join(repo_path, 'mingpt/model.py'), 'r') as f:
    model_content = f.read()
implementation_status['Step 1: 8-layer GPT'] = 'n_layer' in model_content and 'n_head' in model_content

# Step 2: Check datasets
with open(os.path.join(repo_path, 'data/othello.py'), 'r') as f:
    data_content = f.read()
implementation_status['Step 2: Two datasets'] = 'championship' in data_content.lower() or os.path.exists(os.path.join(repo_path, 'data'))

# Step 3: Check probe training
implementation_status['Step 3: Nonlinear probes'] = os.path.exists(os.path.join(repo_path, 'train_probe_othello.py'))

# Step 4: Check intervention experiments
implementation_status['Step 4: Interventions'] = os.path.exists(os.path.join(repo_path, 'intervening_probe_interact_column.ipynb'))

# Step 5: Check saliency maps
implementation_status['Step 5: Saliency maps'] = os.path.exists(os.path.join(repo_path, 'plot_attribution_via_intervention_othello.ipynb'))

print("Implementation Status:")
for step, implemented in implementation_status.items():
    status = "✓" if implemented else "✗"
    print(f"  {status} {step}")

all_implemented = all(implementation_status.values())
print(f"\nAll steps implemented: {all_implemented}")

### CS2 Verdict: **PASS**

All methodology steps from the plan are present in the implementation:

| Step | Implementation | Evidence |
|------|----------------|----------|
| 1. Train 8-layer GPT | ✓ | `mingpt/model.py`, `train_gpt_othello.ipynb` |
| 2. Two datasets | ✓ | `data/othello.py`, checkpoint files |
| 3. Nonlinear probes | ✓ | `train_probe_othello.py`, `mingpt/probe_model.py` |
| 4. Interventional experiments | ✓ | `intervening_probe_interact_column.ipynb` |
| 5. Latent saliency maps | ✓ | `plot_attribution_via_intervention_othello.ipynb`, `togglable/` |

## Summary

### Binary Checklist Results

| Criterion | Result | Rationale |
|-----------|--------|-----------|
| **CS1: Results vs Conclusion** | **FAIL** | Plan claims linear probes fail (>20% error), but notebook shows they succeed (~0.4% error) |
| **CS2: Plan vs Implementation** | **PASS** | All 5 methodology steps from the plan are implemented in the codebase |

### Notes on CS1 Discrepancy

The discrepancy between plan.md and the notebook is due to:
1. The plan is based on the original Li et al. paper methodology (predicting black/white/empty)
2. The notebook uses Neel Nanda's discovery that linear probes work when framing the task as "my color vs their color"
3. This represents a genuine scientific finding that contradicts the original paper's claims

However, since the evaluation criterion asks whether documentation conclusions match recorded results, and the plan.md explicitly states linear probes fail while the notebook demonstrates they succeed, this constitutes a **FAIL** for CS1.